### LCEL (Langchain Expression Language)

**LangChain Expression Language (LCEL) is a declarative way to compose and chain AI building blocks like prompts, models, and output parsers using the pipe (|) operator.** 

* **Key Features of LCEL**
    - *Composability*: Combine small, reusable components into complex pipelines.
    - *Uniform Interface*: All components implement the LangChain Runnable Protocol, supporting .invoke(), .stream(), and .batch() methods out of the box.
    - *Built-in Performance*: Native support for asynchronous operations, parallel execution, streaming, and automatic fallbacks.
    - *Clean Syntax*: Replaces older, complex chain classes with a simple left-to-right data flow (e.g., prompt | model | parser)

#### Groq LPU

**The Groq Language Processing Unit (LPU) is a specialized AI accelerator chip designed from the ground up to speed up generative AI and large language models (LLMs) with ultra-low latency.**

🔑 *Fast Inferencing*.

**Read more here [Whygroq](https://groq.com)**

In [1]:
import os
import openai
from dotenv import load_dotenv

## This function will load all the variable from .env file and will make them available
## os.environ directory (env_variablea)
load_dotenv()

openai.api_key = os.getenv("OPENAI_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

In [2]:
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI

model = ChatGroq(model="openai/gpt-oss-20b",
                    groq_api_key=groq_api_key)

model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.17'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x1128c3620>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x112a44440>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain_core.messages import HumanMessage, SystemMessage

message = [
    SystemMessage(content="Translate the following from English to Spanish"),
    HumanMessage(content="Hello, How are you")
]

result = model.invoke(message)

##### **🔑 Note: In case If you get deprecation warning, Kindly look for the documentation and available replacement.**

In [4]:
## In case, if you are interested only with the content
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
parser.invoke(result)

'Hola, ¿cómo estás?'

In [5]:
## Using LCEL... chain the components
chain = model | parser
chain.invoke(message)

'¡Hola! ¿Cómo estás?'

In [6]:
## prompt templates
from langchain_core.prompts import ChatPromptTemplate

generic_template = "Translate tyhe foloowing into {language} :"

prompt = ChatPromptTemplate.from_messages(
    [('system', generic_template),
     ('user',"{text}")]
)

In [7]:
result = prompt.invoke({'language':'Spanish', 'text':'Hello'})
result.to_messages()

[SystemMessage(content='Translate tyhe foloowing into Spanish :', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [8]:
chain = prompt | model | parser
chain.invoke({'language':'Spanish', 'text':'Hello'})

'Hola'

### LangServe

**LangServe is an open-source library that helps developers quickly deploy LangChain runnables and chains as production-ready REST APIs.** 

* **Key Features**
    - *FastAPI Integration:* Built on top of FastAPI to offer high performance and async support.
    - *Automatic Validation*: Uses Pydantic to validate input and output data automatically.
    - *Built-in Endpoints*: Creates standard routes like /invoke, /batch, /stream, and /stream_log for your AI apps.
    - *Interactive Playground*: Provides a browser-based UI to test prompts and see real-time outputs without writing frontend code. 
    
* **How It Works**
    LangServe wraps your existing LangChain logic into a web server with minimal configuration. It infers schemas directly from your code, generates Swagger documentation, and supports client libraries in Python and JavaScript.